# Substructure and SMARTS search

This notebook uses SMSD 8.2.1 or later to search a small inline library of common drugs
and benzodiazepines with molecule queries and SMARTS patterns. It covers a
first search, hydrogen counts, functional-group flags, enumeration, chemistry
options, limits, corpus decisions and drawing a hit. Every result keeps its
status, so a search that stopped early is never read as an absence. pandas is
optional: without it, the table is printed as plain rows.

## Setup

Run the next cell first in Colab or local Jupyter. It installs SMSD Pro and
the notebook's missing table or numerical dependencies into the current
kernel, using that kernel's Python. An already installed SMSD 8.2.1 or later
is kept. Installation needs internet access; the chemistry examples use inline
inputs. No wheel upload or copied setup cell is needed.

If an older SMSD is already imported, restart the kernel before running
setup. See [Installation](https://asad.github.io/SMSD/HOWTO-INSTALL/).


In [ ]:
# Run this cell first in Colab or a local Jupyter kernel.
import importlib
import importlib.metadata
import importlib.util
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Select a Python 3.12 or later kernel for SMSD Pro wheels")

try:
    _smsd_version = importlib.metadata.version("smsd")
except importlib.metadata.PackageNotFoundError:
    _smsd_version = "0.0.0"
_smsd_parts = tuple(int(part) for part in _smsd_version.split(".")[:3]
                    if part.isdigit())
_requirements = []
if importlib.util.find_spec("smsd") is None or _smsd_parts < (8, 2, 1):
    if "smsd" in sys.modules:
        raise RuntimeError("Restart the kernel, then run this setup cell before importing SMSD")
    _requirements.append("smsd>=8.2.1,<9")
if _requirements:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:",
         *_requirements], check=True,
    )
    importlib.invalidate_caches()
_optional_packages = {'pandas': 'pandas>=2.0,<4'}
_optional_requirements = [_package for _module, _package in _optional_packages.items()
                          if importlib.util.find_spec(_module) is None]
if _optional_requirements:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:",
             *_optional_requirements], check=True,
        )
    except subprocess.CalledProcessError:
        print("Some optional packages have no compatible wheel or could not be downloaded. "
              "The affected cells will report what is missing; SMSD remains available.")
    importlib.invalidate_caches()
import smsd
print("Using SMSD", smsd.__version__)


In [ ]:
# Responsive figures with a caption, an accessible image and an SVG fallback.
import base64
from html import escape


def show_svg(svg_text, description, max_width=800):
    image_uri = "data:image/svg+xml;base64," + base64.b64encode(
        svg_text.encode("utf-8")).decode("ascii")
    figure = (
        f'<figure style="margin:1em 0;max-width:{int(max_width)}px">'
        f'<img src="{image_uri}" alt="{escape(description, quote=True)}" '
        'style="display:block;width:100%;height:auto;box-sizing:border-box;'
        'background:#fff;border:1px solid #d5dce3;border-radius:8px;padding:16px">'
        f'<figcaption style="margin-top:0.6em;line-height:1.5">{escape(description)}</figcaption>'
        '</figure>'
    )
    try:
        from IPython.display import display
    except ImportError:
        print(description)
    else:
        display({"text/html": figure, "image/svg+xml": svg_text,
                 "text/plain": description}, raw=True,
                metadata={"image/svg+xml": {"alt": description}})


def show_table(frame):
    try:
        from IPython.display import display
    except ImportError:
        print(frame.to_string())
    else:
        display(frame)


In [ ]:
# Uses the SMSD package prepared by the setup cell.
import re

import smsd

_release = tuple(int(part) for part in re.findall(r"\d+", smsd.__version__)[:3])
if _release < (8, 2, 1):
    raise RuntimeError("Use SMSD 8.2.1 or later in this notebook's kernel")
print("Using SMSD", smsd.__version__)

try:
    import pandas
except ImportError:
    pandas = None
    print("pandas is not installed; the table is printed as plain rows.")

In [ ]:
drugs = {
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "salicylic acid": "OC(=O)c1ccccc1O",
    "paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "naproxen": "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "phenol": "Oc1ccccc1",
}
benzodiazepines = {
    "diazepam": "CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21",
    "nordazepam": "O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1",
    "temazepam": "CN1C(=O)C(O)N=C(c2ccccc2)c2cc(Cl)ccc21",
    "oxazepam": "OC1N=C(c2ccccc2)c2cc(Cl)ccc2NC1=O",
    "lorazepam": "OC1N=C(c2ccccc2Cl)c2cc(Cl)ccc2NC1=O",
    "clonazepam": "O=C1CN=C(c2ccccc2Cl)c2cc([N+](=O)[O-])ccc2N1",
    "nitrazepam": "O=C1CN=C(c2ccccc2)c2cc([N+](=O)[O-])ccc2N1",
    "flunitrazepam": "CN1C(=O)CN=C(c2ccccc2F)c2cc([N+](=O)[O-])ccc21",
    "bromazepam": "O=C1CN=C(c2ccccn2)c2cc(Br)ccc2N1",
    "tetrazepam": "CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21",
    "alprazolam": "Cc1nnc2n1-c1ccc(Cl)cc1C(c1ccccc1)=NC2",
    "triazolam": "Cc1nnc2n1-c1ccc(Cl)cc1C(c1ccccc1Cl)=NC2",
    "midazolam": "Cc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2",
    "clobazam": "CN1C(=O)CC(=O)N(c2ccccc2)c2cc(Cl)ccc21",
}
library = {**drugs, **benzodiazepines}
print(len(drugs), "common drugs and", len(benzodiazepines), "benzodiazepines")

In [ ]:
# Phenol as a molecule query, searched in 4-tert-butylphenol
query = smsd.Substructure.find("c1ccccc1O")
result = query.match("CC(C)(C)c1ccc(O)cc1")
print("found:", result.found)
print("status:", result.status.name, "optimal:", result.optimal, "complete:", result.complete)
print("query atom -> target atom:", sorted(result.best.atom_mapping.items()))
if not (result.found and result.status == smsd.SearchStatus.COMPLETED):
    raise RuntimeError("Phenol should be found in 4-tert-butylphenol")

## Hydrogen counts

A molecule query does not compare hydrogen counts, so phenol as a molecule
query also matches the ester oxygen of aspirin. A SMARTS `H` primitive, as in
`c[OX2H]`, fixes the count. `to_smarts(..., include_h_count=True)` fixes the
count on every atom, ring positions included, so its pattern matches
unsubstituted phenol only.

In [ ]:
aspirin = drugs["aspirin"]
print("phenol molecule query in aspirin:", smsd.Substructure.find("c1ccccc1O").match(aspirin).found)
print("c[OX2H] in aspirin:", smsd.Substructure.smarts("c[OX2H]").match(aspirin).found)

pattern = smsd.to_smarts("c1ccccc1O", include_h_count=True)
print(pattern)
fixed = smsd.Substructure.smarts(pattern)
for name in ("aspirin", "phenol", "paracetamol"):
    print(f"  {name}: {fixed.match(drugs[name]).found}")

In [ ]:
# Flag four functional groups across the common drugs with one prepared corpus
groups = {
    "carboxylic acid": "C(=O)[OX2H1]",
    "phenol": "c[OX2H]",
    "aryl ester": "cOC(=O)[#6]",
    "amide": "[#6](=O)[#7X3]",
}
corpus = smsd.TargetCorpus.from_smiles(list(drugs.values()))
table = {"name": list(drugs)}
for label, pattern in groups.items():
    results = smsd.Substructure.smarts(pattern).match_many(corpus, num_threads=2)
    if any(result.status != smsd.SearchStatus.COMPLETED for result in results):
        raise RuntimeError(f"The {label} search stopped early, so a False is not proved")
    table[label] = [result.found for result in results]

if pandas is not None:
    show_table(pandas.DataFrame(table).set_index("name"))
else:
    print(tuple(table))
    for row in zip(*table.values()):
        print(row)

In [ ]:
# Every embedding of benzene in toluene, with a limit and a time limit
benzene = smsd.Substructure.find("c1ccccc1").with_timeout(1000)
every = benzene.match_all("Cc1ccccc1", limit=100)
occurrences = {frozenset(match.atom_mapping.values()) for match in every.matches}
print("embeddings:", len(every.matches), every.status.name, "complete:", every.complete)
print("distinct ring occurrences:", len(occurrences))

capped = benzene.match_all("Cc1ccccc1", limit=5)
print("limit 5:", len(capped.matches), capped.status.name,
      "optimal:", capped.optimal, "complete:", capped.complete)

## Chemistry options

`options.chemistry` decides which atoms and bonds may match. The default
`FLEXIBLE` aromaticity lets aromatic atoms and bonds of the query match
non-aromatic ones, so the benzodiazepine core below also finds tetrazepam,
whose phenyl ring is replaced by a cyclohexenyl ring. `STRICT` aromaticity
excludes it. The last two rows show formal-charge and ring-membership matching
on small molecules. Options objects take no keyword arguments: set each field
as an attribute.

In [ ]:
core = "O=C1CN=C(c2ccccc2)c2ccccc2N1"   # 5-phenyl-1,4-benzodiazepin-2-one
strict = smsd.Substructure.Options()
strict.chemistry.aromaticity_mode = smsd.AromaticityMode.STRICT
names, targets = list(benzodiazepines), list(benzodiazepines.values())
hits = {}
for label, search in (("default", smsd.Substructure.find(core)),
                      ("STRICT aromaticity", smsd.Substructure.find(core).with_options(strict))):
    results = search.match_many(targets, num_threads=2)
    if any(result.status != smsd.SearchStatus.COMPLETED for result in results):
        raise RuntimeError("A core search stopped early")
    hits[label] = [name for name, result in zip(names, results) if result.found]
    print(f"{label}: {len(hits[label])} hits")
print("found only with the default:", sorted(set(hits["default"]) - set(hits["STRICT aromaticity"])))


def found(query, target, **fields):
    options = smsd.Substructure.Options()
    for name, value in fields.items():
        setattr(options.chemistry, name, value)
    return smsd.Substructure.find(query).with_options(options).match(target).found


print("C[NH3+] in methylamine, match_formal_charge:",
      found("C[NH3+]", "CN"), "->", found("C[NH3+]", "CN", match_formal_charge=True))
print("propane in cyclohexane, ring_matches_ring_only:",
      found("CCC", "C1CCCCC1"), "->", found("CCC", "C1CCCCC1", ring_matches_ring_only=True))

In [ ]:
# A stopped search with no matches proves nothing: optimal and complete are False
query = smsd.Substructure.find("c1ccccc1O")
target = "CC(C)(C)c1ccc(O)cc1"


def report(label, result):
    print(f"{label:14}", result.status.name, "found:", result.found,
          "optimal:", result.optimal, "complete:", result.complete)


report("timeout 0", query.with_timeout(0).match(target))

options = smsd.Substructure.Options()
options.node_limit = 3
report("node limit 3", query.with_options(options).match(target))

token = smsd.CancellationToken()
options = smsd.Substructure.Options()
options.cancellation = token
options.timeout_ms = 1000
token.cancel()
report("cancelled", query.with_options(options).match(target))

In [ ]:
# One decision per slot: targets that cannot contain the core are ruled out
# before matching, with a proof; the rest are searched as usual.
library_corpus = smsd.TargetCorpus.from_smiles(list(library.values()))
decisions = smsd.Substructure.find(core).match_corpus(library_corpus, num_threads=2)
counts = {}
for name, decision in zip(library, decisions):
    if decision.result is None:
        outcome = "ruled out before matching"
    elif decision.threshold_reached:
        outcome = "found"
    elif decision.proven_below_threshold:
        outcome = "searched, proved absent"
    else:
        outcome = "unknown (search stopped)"
    counts[outcome] = counts.get(outcome, 0) + 1
    print(f"{name:15} {outcome}")
print(counts)

In [ ]:
# Draw paracetamol with its aromatic hydroxyl highlighted in mint. substructure_svg
# takes the pattern and the target as searched and keeps the search status.
from smsd.highlight import substructure_svg

paracetamol = drugs["paracetamol"]
pattern = "c[OX2H]"
hit = smsd.Substructure.smarts(pattern).match(paracetamol)
print("hit atoms in paracetamol:", sorted(hit.best.atom_mapping.values()), hit.status.name)
svg = substructure_svg(pattern, paracetamol, hit)

show_svg(
    svg,
    "Paracetamol with the aromatic carbon and hydroxyl oxygen matched by "
    "SMARTS c[OX2H] highlighted in mint.",
    max_width=560,
)


## Summary

Molecule queries and SMARTS patterns share one prepared-query interface, and
every result states whether the search completed. Check the status before
reading a missing match as an absence, give enumerations a limit and a time
limit, and use SMARTS `H` primitives when hydrogen counts matter.

For the full reference, see the
[Substructure search](https://asad.github.io/SMSD/guide/SUBSTRUCTURE/) and
[SMARTS queries](https://asad.github.io/SMSD/guide/SMARTS/) chapters of the user guide.